## RuToxic Dataset


In [ ]:
from __future__ import annotations

import html
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "data" else NOTEBOOK_DIR
DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
INTERIM_ROOT = DATA_ROOT / "interim"
PROCESSED_ROOT = DATA_ROOT / "processed"
for folder in (RAW_ROOT, INTERIM_ROOT, PROCESSED_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'RuToxic'
DEFAULT_SOURCE = '../data/raw/rutoxic'
LOCAL_CANDIDATES = [
    '../data/raw/rutoxic.csv',,
'../data/raw/rutoxic.parquet',,
'../data/raw/rutoxic/train.csv',
]
LABEL_COLUMNS = ['toxic', 'label', 'class']
LANGUAGE_HINT = 'ru'

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)


In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
USER_RE = re.compile(r"@[\w_]+")
HTML_RE = re.compile(r"<[^>]+>")
MULTISPACE_RE = re.compile(r"\s+")
PUNCT_RE = re.compile(r"[^\w\s!?.,]+", flags=re.UNICODE)


def _read_tabular(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".csv", ".tsv"}:
        sep = "	" if path.suffix == ".tsv" else ","
        return pd.read_csv(path, sep=sep)
    raise ValueError(f"Unsupported file: {path}")


def load_text_dataset() -> pd.DataFrame:
    for candidate in [Path(path) for path in LOCAL_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.is_dir():
            train_csv = next(candidate.rglob("*.csv"), None)
            train_parquet = next(candidate.rglob("*.parquet"), None)
            if train_parquet:
                frame = _read_tabular(train_parquet)
                frame["source"] = str(train_parquet)
                return frame
            if train_csv:
                frame = _read_tabular(train_csv)
                frame["source"] = str(train_csv)
                return frame
            continue

        frame = _read_tabular(candidate)
        frame["source"] = str(candidate)
        return frame

    raise FileNotFoundError(
        "Dataset was not found locally. Export the Kaggle or HuggingFace dataset into ../data/raw/ and rerun."
    )


def standardize_text_frame(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    rename = {}
    for column in df.columns:
        low = column.lower()
        if low in {"comment_text", "text", "content", "sentence"}:
            rename[column] = "text"
        elif low in {"lang", "language", "locale"}:
            rename[column] = "language"
        elif low in {"split", "subset"}:
            rename[column] = "split"
    df = df.rename(columns=rename)

    if "text" not in df.columns:
        text_like = [column for column in df.columns if "text" in column.lower() or "comment" in column.lower()]
        if not text_like:
            raise KeyError("A text column was not found.")
        df["text"] = df[text_like[0]]

    if "language" not in df.columns:
        df["language"] = LANGUAGE_HINT

    label_candidates = [column for column in df.columns if column.lower() in LABEL_COLUMNS]
    if label_candidates:
        primary = label_candidates[0]
        df["label"] = pd.to_numeric(df[primary], errors="coerce").fillna(0.0)
    else:
        toxic_like = [
            column
            for column in df.columns
            if any(key in column.lower() for key in ["toxic", "obscene", "insult", "threat", "hate"])
        ]
        if toxic_like:
            df["label"] = (
                df[toxic_like]
                .apply(pd.to_numeric, errors="coerce")
                .fillna(0.0)
                .max(axis=1)
            )
        else:
            df["label"] = 0

    df["label"] = (df["label"] >= 0.5).astype(int)
    if "split" not in df.columns:
        df["split"] = "train"
    return df[["text", "label", "language", "split", "source"]].copy()


raw_df = load_text_dataset()
df = standardize_text_frame(raw_df)
print("Rows:", len(df))
display(df.head())


In [ ]:
df["char_len"] = df["text"].astype(str).str.len()
df["word_len"] = df["text"].astype(str).str.split().str.len()
df["uppercase_ratio"] = df["text"].astype(str).map(lambda value: sum(ch.isupper() for ch in value) / max(len(value), 1))
df["url_count"] = df["text"].astype(str).map(lambda value: len(URL_RE.findall(value)))

token_counter = Counter()
for text in df["text"].astype(str).head(min(5000, len(df))):
    token_counter.update(re.findall(r"\w+", text.lower()))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.countplot(data=df, x="label", ax=axes[0, 0])
sns.histplot(data=df, x="char_len", hue="label", bins=30, ax=axes[0, 1], element="step")
sns.histplot(data=df, x="word_len", hue="label", bins=30, ax=axes[0, 2], element="step")
sns.boxplot(data=df, x="label", y="uppercase_ratio", ax=axes[1, 0])
sns.boxplot(data=df, x="label", y="url_count", ax=axes[1, 1])
language_counts = df["language"].astype(str).value_counts().head(10).rename_axis("language").reset_index(name="count")
sns.barplot(data=language_counts, x="count", y="language", ax=axes[1, 2])
plt.tight_layout()
print("Top tokens:", token_counter.most_common(25))


In [ ]:
def normalize_text(text: str) -> str:
    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace(" ", " ")
    text = URL_RE.sub(" <URL> ", text)
    text = USER_RE.sub(" <USER> ", text)
    text = HTML_RE.sub(" ", text)
    text = PUNCT_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text.lower()


def enrich_features(frame: pd.DataFrame) -> pd.DataFrame:
    clean = frame.copy()
    clean["clean_text"] = clean["text"].map(normalize_text)
    clean["clean_char_len"] = clean["clean_text"].str.len()
    clean["clean_word_len"] = clean["clean_text"].str.split().str.len()
    clean["exclamation_count"] = clean["text"].astype(str).str.count("!")
    clean["question_count"] = clean["text"].astype(str).str.count("\?")
    clean["digit_ratio"] = clean["text"].astype(str).map(lambda value: sum(ch.isdigit() for ch in value) / max(len(value), 1))
    return clean


clean_df = enrich_features(df)
clean_df = clean_df[clean_df["clean_word_len"].fillna(0) > 0].copy()
clean_df = clean_df.drop_duplicates(subset=["clean_text", "label"]).reset_index(drop=True)

train_df, valid_df = train_test_split(
    clean_df,
    test_size=0.2 if len(clean_df) >= 50 else 0.3,
    stratify=clean_df["label"] if clean_df["label"].nunique() > 1 else None,
    random_state=42,
)
train_df["split"] = "train"
valid_df["split"] = "valid"
clean_df = pd.concat([train_df, valid_df], ignore_index=True)


In [ ]:
baseline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2 if len(train_df) > 100 else 1,
                max_df=0.95,
                sublinear_tf=True,
            ),
        ),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

cv = cross_validate(
    baseline,
    train_df["clean_text"],
    train_df["label"],
    scoring=["precision", "recall", "f1", "roc_auc"],
    cv=3 if train_df["label"].value_counts().min() >= 3 else 2,
    n_jobs=-1,
)
print({metric: round(float(np.mean(values)), 4) for metric, values in cv.items() if metric.startswith("test_")})

baseline.fit(train_df["clean_text"], train_df["label"])
valid_scores = baseline.predict_proba(valid_df["clean_text"])[:, 1]
valid_pred = (valid_scores >= 0.5).astype(int)

print(classification_report(valid_df["label"], valid_pred, digits=4))
if len(np.unique(valid_df["label"])) > 1:
    print("ROC AUC:", round(roc_auc_score(valid_df["label"], valid_scores), 4))
print("Confusion matrix:", confusion_matrix(valid_df["label"], valid_pred))


In [ ]:
export_path = PROCESSED_ROOT / 'rutoxic'
export_path = export_path.with_suffix(".parquet")
clean_df.to_parquet(export_path, index=False)

profile = {
    "dataset": DATASET_NAME,
    "rows": int(len(clean_df)),
    "class_balance": clean_df["label"].value_counts(normalize=True).to_dict(),
    "languages": clean_df["language"].value_counts().head(10).to_dict(),
    "avg_lengths": clean_df[["clean_char_len", "clean_word_len"]].mean().round(2).to_dict(),
}
profile_path = INTERIM_ROOT / 'rutoxic'
profile_path = profile_path.with_suffix(".json")
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
profile
